In [1]:
import numpy as np
import pandas as pd
import os
import pathlib
import matplotlib.pyplot as plt
from tqdm import tqdm
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor
import concurrent.futures

import scipy.special as sp
import yaml
import pathlib
from pathlib import Path
import shutil

In [2]:
#--------------------------------------------------------------------------------------------------------------------------
# setting
#--------------------------------------------------------------------------------------------------------------------------

data_name = "Default"

if_triple_bin = True  # True for triple bin integrations for all fixed target exps

#--------------------------------------------------------------------------------------------------------------------------
# read path
#--------------------------------------------------------------------------------------------------------------------------

abs_y_list = [
    "CMS7", "CMS8",
    "CMS13-00y04", "CMS13-04y08", "CMS13-08y12", "CMS13-12y16", "CMS13-16y24",
    "CMS13-50Q76", "CMS13-76Q106", "CMS13-106Q170", "CMS13-170Q350", "CMS13-350Q1000",
    "ATLAS7-00y10", "ATLAS7-10y20", "ATLAS7-20y24",
    "ATLAS8-46Q66", "ATLAS8-116Q150",
    "ATLAS13",
    "ATLAS8-00y04", "ATLAS8-04y08", "ATLAS8-08y12", "ATLAS8-12y16", "ATLAS8-16y20", "ATLAS8-20y24",
    "STAR510"
]
#--------------------------------------------------------------------------------------------------------------------------
# read path
#--------------------------------------------------------------------------------------------------------------------------

before_dir = Path("Raw")

#--------------------------------------------------------------------------------------------------------------------------
# write path
#--------------------------------------------------------------------------------------------------------------------------

after_dir = Path(f"{data_name}/Processed")

Auxilary Functions

DY

In [3]:
experiments =[
    'ATLAS_7',
    'ATLAS_8', 
    'ATLAS_13', 
    'CDF_I',
    'CDF_II',
    'CMS_7',
    'CMS_8',
    'CMS_13',    
    'D0_I',
    'D0_II',
    'D0_II_mu',
    'E288',
    'E605',
    'E772',
    'LHCb_7',
    'LHCb_8',
    'LHCb_13',    
    'PHENIX',
    'STAR'
]
excludes = []

In [4]:
processes = ["DY","SIDIS"]

for process in processes:
    process_dir = after_dir / process
    if process_dir.exists():
        shutil.rmtree(process_dir) 
    process_dir.mkdir(parents=True, exist_ok=True)

Collider

In [5]:
total_xsec_added = []
abs_y_added = []

for experiment in experiments:

    exp_dir = before_dir / "DY" / experiment
    file_names = [file.stem for file in sorted(exp_dir.glob("*.csv"))]
    file_paths = [exp_dir / f"{name}.csv" for name in file_names] 

    save_dir = Path(after_dir) / "DY" / experiment
    save_dir.mkdir(parents=True, exist_ok=True)

    for file_name in (f for f in file_names if f not in excludes):

        file_path = exp_dir / f"{file_name}.csv"

        df = pd.read_csv(file_path)
        N = len(df["xsec"])

        df_processed = df

        if df_processed["observable"][0] in ["1/σ*dσ/dqT","dσ/dqT"]:

            # Change qT = 1e-5 to 0
            if df_processed["qT_min"][0] == 1e-5:
                df_processed["qT_min"][0] = 0.0

            N = len(df["xsec"])
            bin_integrate_factors = np.ones(N)

            # Handle |y| 
            if file_name in abs_y_list:
                bin_integrate_factors = bin_integrate_factors * 2.0
                abs_y_added.append(file_name)         
            
            # Divide by qT bin
            if df_processed["qT_if_integrate"][0]:
                qT_delta = df["qT_max"] - df["qT_min"]
                bin_integrate_factors = bin_integrate_factors/qT_delta

            df_processed["bin_integrate_factor"] = bin_integrate_factors

            # save
            destination = after_dir / "DY" / experiment / f"{file_name}.csv"
            if destination.exists():
                raise FileExistsError(f"{file_name} already exists")
            df_processed.round(10).to_csv(destination, index=False)
            
            print(f"{file_name} generated")

print("Added total cross-sections to")
display(total_xsec_added)
print("Added |y| flag to")
display(abs_y_added)

ATLAS7-00y10 generated
ATLAS7-10y20 generated
ATLAS7-20y24 generated
ATLAS8-00y04 generated
ATLAS8-04y08 generated
ATLAS8-08y12 generated
ATLAS8-116Q150 generated
ATLAS8-12y16 generated
ATLAS8-16y20 generated
ATLAS8-20y24 generated
ATLAS8-46Q66 generated
ATLAS13 generated
CDF1 generated
CDF2 generated
CMS7 generated
CMS8 generated
CMS13-00y04 generated
CMS13-04y08 generated
CMS13-08y12 generated
CMS13-106Q170 generated
CMS13-12y16 generated
CMS13-16y24 generated
CMS13-170Q350 generated
CMS13-350Q1000 generated
D01 generated
D02 generated
D02mu generated
LHCb7 generated
LHCb8 generated
LHCb13 generated
PHENIX200 generated
STAR510 generated
Added total cross-sections to


[]

Added |y| flag to


['ATLAS7-00y10',
 'ATLAS7-10y20',
 'ATLAS7-20y24',
 'ATLAS8-00y04',
 'ATLAS8-04y08',
 'ATLAS8-08y12',
 'ATLAS8-116Q150',
 'ATLAS8-12y16',
 'ATLAS8-16y20',
 'ATLAS8-20y24',
 'ATLAS8-46Q66',
 'ATLAS13',
 'CMS7',
 'CMS8',
 'CMS13-00y04',
 'CMS13-04y08',
 'CMS13-08y12',
 'CMS13-106Q170',
 'CMS13-12y16',
 'CMS13-16y24',
 'CMS13-170Q350',
 'CMS13-350Q1000',
 'STAR510']

Fixed Targets

In [6]:
total_xsec_added = []
abs_y_added = []

for experiment in experiments:

    exp_dir = before_dir / "DY" / experiment
    file_names = [file.stem for file in sorted(exp_dir.glob("*.csv"))]
    file_paths = [exp_dir / f"{name}.csv" for name in file_names] 

    save_dir = Path(after_dir) / "DY" / experiment
    save_dir.mkdir(parents=True, exist_ok=True)

    for file_name in (f for f in file_names if f not in excludes):

        file_path = exp_dir / f"{file_name}.csv"

        df = pd.read_csv(file_path)
        N = len(df["xsec"])

        df_processed = df

        if df_processed["observable"][0] in ["E*dσ/d3q","E*dσ/d3q(xf)"]:
            
            N = len(df["xsec"])
            bin_integrate_factors = np.ones(N)

            if df_processed["qT_if_integrate"][0]:
                qT_delta = df["qT_max"] - df["qT_min"]
                bin_integrate_factors = bin_integrate_factors/qT_delta
            #if df_processed["Q_if_integrate"][0]:
            #    Q_delta = df["Q_max"] - df["Q_min"]
            #    bin_integrate_factors = bin_integrate_factors/Q_delta
            if df_processed["y_if_integrate"][0]:
                if df_processed["observable"][0] == "E*dσ/d3q":
                    y_delta = df["y_max"] - df["y_min"]
                    bin_integrate_factors = bin_integrate_factors/y_delta
                if df_processed["observable"][0] == "E*dσ/d3q(xf)":
                    xf_delta = df["xf_max"] - df["xf_min"]
                    bin_integrate_factors = bin_integrate_factors/xf_delta
                    
            df_processed["bin_integrate_factor"] = bin_integrate_factors

            # save
            destination = after_dir / "DY" / experiment / f"{file_name}.csv"
            if destination.exists():
                raise FileExistsError(f"{file_name} already exists")
            df_processed.round(10).to_csv(destination, index=False)

            print(f"{file_name} generated")

E228-200-4Q5 generated
E228-200-5Q6 generated
E228-200-6Q7 generated
E228-200-7Q8 generated
E228-200-8Q9 generated
E228-300-11Q12 generated
E228-300-4Q5 generated
E228-300-5Q6 generated
E228-300-6Q7 generated
E228-300-7Q8 generated
E228-300-8Q9 generated
E228-400-11Q12 generated
E228-400-12Q13 generated
E228-400-13Q14 generated
E228-400-5Q6 generated
E228-400-6Q7 generated
E228-400-7Q8 generated
E228-400-8Q9 generated
E605-10.5Q11.5 generated
E605-11.5Q13.5 generated
E605-13.5Q18 generated
E605-7Q8 generated
E605-8Q9 generated
E772-11Q12 generated
E772-12Q13 generated
E772-13Q14 generated
E772-14Q15 generated
E772-5Q6 generated
E772-6Q7 generated
E772-7Q8 generated
E772-8Q9 generated
